# Correlated Subqueries and EXISTS

## Overview
A **correlated subquery** references a column from the outer query. Unlike a non-correlated subquery (which runs once), a correlated subquery re-executes for every row of the outer query — once per row.

```sql
-- Correlated: the subquery references e.patient_id from the outer query
SELECT *
FROM   encounters AS e
WHERE  cost_cad > (
    SELECT AVG(cost_cad)
    FROM   encounters
    WHERE  patient_id = e.patient_id   -- references outer row
);
```

**Performance:** Correlated subqueries are O(n × m) in the naive case. Most modern optimisers can detect and transform simple cases, but for large tables, a window function or join is almost always faster.

### EXISTS and NOT EXISTS
`EXISTS` tests whether the subquery returns **any rows at all** — it short-circuits on the first match. It does not care about the values returned, only whether any row exists.

```sql
-- Patients who have had at least one admission
WHERE EXISTS (
    SELECT 1 FROM encounters
    WHERE patient_id = p.patient_id AND admitted = 1
)

-- Patients who have NEVER been admitted
WHERE NOT EXISTS (
    SELECT 1 FROM encounters
    WHERE patient_id = p.patient_id AND admitted = 1
)
```

**NOT EXISTS vs NOT IN:** `NOT IN` fails silently when the subquery returns any NULL (the result is always NULL/unknown). `NOT EXISTS` handles NULLs correctly and is almost always the right choice for exclusion logic.

---

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, active INTEGER DEFAULT 1);
CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT, province TEXT, manager_id INTEGER);
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_id INTEGER);
CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER, dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER);
CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT, category TEXT);
CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER, test_name TEXT, result_val REAL, units TEXT, collected TEXT, flagged INTEGER);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),
  (5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),
  (7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),
  (9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),
  (11,'Diana','Flores','2000-02-14','F','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),(11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Osei','Family Medicine','ON',10),(13,'Dr. Ethan Larson','Orthopaedics','QC',10),
  (14,'Dr. Priya Sharma','Emergency','AB',10),(15,'Dr. Lucas Petit','Radiology','QC',11);
INSERT INTO departments VALUES
  (1,'Emergency','Tower A',14),(2,'Cardiology','Tower B',10),(3,'Oncology','Tower C',11),
  (4,'Family Medicine','Clinic',12),(5,'Orthopaedics','Tower A',13),
  (6,'Radiology','Tower B',15),(7,'ICU','Tower C',NULL);
INSERT INTO encounters VALUES
  (1,1,10,2,'2023-01-15','I25.1',3200.00,1),(2,2,14,1,'2023-02-20','J18.9',1850.00,1),
  (3,3,12,4,'2023-03-05','Z00.0',120.00,0),(4,4,13,5,'2023-03-18','M17.1',5500.00,1),
  (5,5,12,4,'2023-04-02','J06.9',95.00,0),(6,6,14,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,2,'2023-06-22','I10',2100.00,1),(8,8,12,4,'2023-07-14',NULL,80.00,0),
  (9,1,14,1,'2023-08-30','R55',410.00,0),(10,3,12,4,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,2,'2023-10-01','I48.0',1750.00,1),(12,10,14,1,'2023-11-03','S52.5',2200.00,1),
  (13,2,10,2,'2023-11-20','I25.1',2900.00,1),(14,6,12,4,'2023-12-01','Z00.0',115.00,0);
INSERT INTO diagnoses VALUES
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),
  ('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),
  ('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),
  ('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');
INSERT INTO lab_results VALUES
  (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),
  (3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),
  (5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),
  (7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),
  (9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),
  (11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);
""")
conn.commit()
print('Healthcare schema ready.')


Healthcare schema ready.


---
## Correlated subquery — per-row comparison

In [4]:
# Encounters costing more than that patient's own average
print("=== Encounters above the patient's own average cost (correlated subquery) ===")
q = """
SELECT  e.enc_id,
        e.patient_id,
        p.first_name || ' ' || p.last_name          AS patient,
        e.enc_date,
        e.cost_cad                                   AS this_cost,
        ROUND((SELECT AVG(cost_cad)
               FROM   encounters
               WHERE  patient_id = e.patient_id), 2) AS patient_avg_cost
FROM    encounters AS e
JOIN    patients   AS p ON e.patient_id = p.patient_id
WHERE   e.cost_cad > (
            SELECT AVG(cost_cad)
            FROM   encounters
            WHERE  patient_id = e.patient_id
        )
ORDER BY e.patient_id, e.enc_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("The subquery re-executes for every row of encounters,")
print("each time filtering to the current row's patient_id.")

=== Encounters above the patient's own average cost (correlated subquery) ===
 enc_id  patient_id          patient   enc_date  this_cost  patient_avg_cost
      1           1      Aroha Ngata 2023-01-15     3200.0            1805.0
     13           2        Liam Chen 2023-11-20     2900.0            2375.0
      3           3 Fatima Al-Rashid 2023-03-05      120.0             115.0
      6           6    Noah Williams 2023-05-11      780.0             447.5

The subquery re-executes for every row of encounters,
each time filtering to the current row's patient_id.


---
## EXISTS — semi-join: does a matching row exist?

In [5]:
# Patients who have had at least one inpatient admission
print('=== Patients with at least one admission (EXISTS) ===')
q = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.province
FROM    patients AS p
WHERE   EXISTS (
    SELECT 1
    FROM   encounters AS e
    WHERE  e.patient_id = p.patient_id
      AND  e.admitted   = 1
)
ORDER BY p.last_name
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Patients who have at least one flagged lab result
print()
print('=== Patients with at least one flagged lab result (EXISTS) ===')
q2 = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient
FROM    patients AS p
WHERE   EXISTS (
    SELECT 1
    FROM   lab_results AS lr
    WHERE  lr.patient_id = p.patient_id
      AND  lr.flagged    = 1
)
ORDER BY p.last_name
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Patients with at least one admission (EXISTS) ===
 patient_id       patient province
          2     Liam Chen       NS
          4 James MacLeod       NB
          9    Priya Nair       BC
          1   Aroha Ngata       NB
         10 Marcus Okafor       ON
          7     Mei Zhang       ON

=== Patients with at least one flagged lab result (EXISTS) ===
 patient_id       patient
          2     Liam Chen
          4 James MacLeod
          9    Priya Nair
         10 Marcus Okafor
          7     Mei Zhang


---
## NOT EXISTS — anti-join: the safer alternative to NOT IN

In [6]:
# Patients who have NEVER had any encounter
print('=== Patients with no encounters (NOT EXISTS) ===')
q = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.active
FROM    patients AS p
WHERE   NOT EXISTS (
    SELECT 1 FROM encounters
    WHERE  patient_id = p.patient_id
)
ORDER BY p.last_name
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Diagnoses in reference table never used in any encounter
print()
print('=== Diagnoses never used in any encounter (NOT EXISTS) ===')
q2 = """
SELECT  d.diag_code, d.description, d.category
FROM    diagnoses AS d
WHERE   NOT EXISTS (
    SELECT 1 FROM encounters
    WHERE  diag_code = d.diag_code
)
ORDER BY d.category, d.diag_code
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# Why NOT IN is dangerous: demonstrate the NULL trap
print()
print('=== NOT IN with NULLs in the list — the silent trap ===')
conn.execute("INSERT INTO encounters VALUES (99,11,14,1,'2023-12-15',NULL,200.00,0)")
conn.commit()
q3 = """
SELECT COUNT(*) AS rows_not_in_null_list
FROM   diagnoses
WHERE  diag_code NOT IN (
    SELECT diag_code FROM encounters   -- includes a NULL diag_code!
)
"""
result = conn.execute(q3.strip()).fetchone()[0]
print(f'NOT IN with a NULL in the subquery returns {result} rows (should be 2+)')
print('Because: diag_code NOT IN (..., NULL) evaluates to NULL for every row.')
print('Use NOT EXISTS instead — it handles NULLs correctly.')
conn.execute("DELETE FROM encounters WHERE enc_id = 99")
conn.commit()


=== Patients with no encounters (NOT EXISTS) ===
 patient_id      patient  active
         11 Diana Flores       1

=== Diagnoses never used in any encounter (NOT EXISTS) ===
diag_code                description       category
    I50.9 Heart failure, unspecified Cardiovascular
    E11.9   Type 2 diabetes mellitus      Endocrine

=== NOT IN with NULLs in the list — the silent trap ===
NOT IN with a NULL in the subquery returns 0 rows (should be 2+)
Because: diag_code NOT IN (..., NULL) evaluates to NULL for every row.
Use NOT EXISTS instead — it handles NULLs correctly.


---
## Correlated subquery for greatest-N-per-group

In [7]:
# Most recent encounter per patient using correlated subquery
# (ROW_NUMBER window function is cleaner — shown for comparison)
print('=== Most recent encounter per patient (correlated subquery approach) ===')
q = """
SELECT  e.enc_id,
        e.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        e.enc_date,
        e.cost_cad
FROM    encounters AS e
JOIN    patients   AS p ON e.patient_id = p.patient_id
WHERE   e.enc_date = (
    SELECT MAX(enc_date)
    FROM   encounters
    WHERE  patient_id = e.patient_id
)
ORDER BY e.enc_date DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Window function equivalent (more efficient for large tables):')
print("""
SELECT enc_id, patient_id, enc_date, cost_cad
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY enc_date DESC, enc_id DESC) AS rn
    FROM encounters
) WHERE rn = 1;
""")

conn.close()


=== Most recent encounter per patient (correlated subquery approach) ===
 enc_id  patient_id          patient   enc_date  cost_cad
     14           6    Noah Williams 2023-12-01     115.0
     13           2        Liam Chen 2023-11-20    2900.0
     12          10    Marcus Okafor 2023-11-03    2200.0
     11           9       Priya Nair 2023-10-01    1750.0
     10           3 Fatima Al-Rashid 2023-09-12     110.0
      9           1      Aroha Ngata 2023-08-30     410.0
      8           8   Ethan Tremblay 2023-07-14      80.0
      7           7        Mei Zhang 2023-06-22    2100.0
      5           5     Sofia Petrov 2023-04-02      95.0
      4           4    James MacLeod 2023-03-18    5500.0

Window function equivalent (more efficient for large tables):

SELECT enc_id, patient_id, enc_date, cost_cad
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY enc_date DESC, enc_id DESC) AS rn
    FROM encounters
) WHERE rn = 1;



---
## Common Pitfalls

**1. NOT IN fails silently when the subquery returns any NULL**
`WHERE diag_code NOT IN (SELECT diag_code FROM encounters)` returns zero rows if any `encounters.diag_code` is NULL — because `'I10' NOT IN (..., NULL)` evaluates to NULL (unknown), not TRUE. Always use `NOT EXISTS` for exclusion logic, or add `WHERE diag_code IS NOT NULL` to the inner subquery.

**2. Correlated subqueries are O(n × m) without optimiser help**
A correlated subquery in WHERE re-executes for every row of the outer query. On a table with 1 million rows, this could mean 1 million subquery executions. The optimiser often converts simple cases to joins, but complex correlated subqueries may not be optimised. For performance-critical code, rewrite as a JOIN or window function.

**3. EXISTS with SELECT 1 vs SELECT * — no performance difference**
`EXISTS (SELECT 1 ...)`, `EXISTS (SELECT * ...)`, and `EXISTS (SELECT col ...)` all behave identically. The database short-circuits on the first matching row regardless of what the subquery selects. Use `SELECT 1` by convention — it makes the intent explicit.

**4. Correlated subquery in SELECT creates implicit one-to-one dependency**
`SELECT (SELECT dept_name FROM departments WHERE dept_id = e.dept_id)` assumes each encounter has at most one matching department. If the subquery returns more than one row it errors. A JOIN is safer and more explicit about the expected cardinality.

**5. NOT EXISTS vs LEFT JOIN IS NULL — both work, style differs**
Both patterns find unmatched rows. `NOT EXISTS` is often more readable for exclusion logic. `LEFT JOIN ... WHERE right.key IS NULL` can be faster in some databases because the planner sees it as an anti-join directly. In PostgreSQL both are optimised to the same plan; choose based on readability.

**6. Forgetting the correlation — accidentally writing a non-correlated subquery**
`WHERE cost_cad > (SELECT AVG(cost_cad) FROM encounters)` computes the global average (non-correlated). `WHERE cost_cad > (SELECT AVG(cost_cad) FROM encounters WHERE patient_id = e.patient_id)` computes the per-patient average (correlated). A missing correlation condition silently changes the semantics of the query.


---
*sql_methods_library - Samantha McGarrigle*